In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.load import loadMatches
from src.split import splitMatches
from src.pitch import distanceToGoal, goalAngle, inShotCone

# here we are lopading the shots parquest

In [2]:
Shots = pd.read_parquet("../data/clean/shots.parquet")

print("Total shots:", len(Shots))
print("Total columns:", len(Shots.columns))
print(Shots[["id", "match_id", "x", "y", "goal"]].head())

Total shots: 99746
Total columns: 24
                                     id  match_id      x     y  goal
0  452ac1fc-6e97-459f-8819-62bc82078292   3895292  107.0  53.4     0
1  345c34d9-2c2f-4cec-860e-3b3704ebc4f3   3895292   96.5  39.9     0
2  6627cd42-a35b-4e4e-9fd3-05d3269bca36   3895292   88.2  45.5     0
3  3d1c53e1-4b07-4648-806c-2dc0b2ee5347   3895292   98.5  49.2     0
4  bcdb8320-5e4f-4c64-910a-f4ff7ace5e8b   3895292  111.2  37.8     0


adding trainng and test split fold

In [3]:
Matches = loadMatches()
Splits = splitMatches(Matches)

Shots = Shots.merge(Splits, on="match_id", validate="many_to_one")

print("Total shots:", len(Shots))
print("Missing folds:", Shots["fold"].isna().sum())
print("Test shots:", Shots["test"].sum())
print("Training shots:", (~Shots["test"]).sum())
print(Shots["fold"].value_counts().sort_index())

Total shots: 99746
Missing folds: 0
Test shots: 24802
Training shots: 74944
fold
0    24802
1    15047
2    14992
3    15056
4    14943
5    14906
Name: count, dtype: int64


# creatingf 1 hot encoded featsure

In [4]:
Categories = [
    "shot_body_part",
    "shot_technique",
    "shot_type",
    "play_pattern"
]

Encoded = pd.get_dummies(Shots[Categories], dtype=int)

Shots = pd.concat([Shots, Encoded], axis=1)

print("One-hot columns created:", len(Encoded.columns))
print("Total columns:", len(Shots.columns))
print(Encoded.head())

One-hot columns created: 24
Total columns: 50
   shot_body_part_Head  shot_body_part_Left Foot  shot_body_part_Other  \
0                    0                         0                     0   
1                    0                         0                     0   
2                    0                         0                     0   
3                    0                         1                     0   
4                    1                         0                     0   

   shot_body_part_Right Foot  shot_technique_Backheel  \
0                          1                        0   
1                          1                        0   
2                          1                        0   
3                          0                        0   
4                          0                        0   

   shot_technique_Diving Header  shot_technique_Half Volley  \
0                             0                           1   
1                             0         

here now calculating dis and angle

In [5]:
Shots["distance"] = distanceToGoal(Shots["x"], Shots["y"])

Shots["angle"] = goalAngle(Shots["x"], Shots["y"])

print(Shots[["x", "y", "distance", "angle"]].head())
print("Missing distances:", Shots["distance"].isna().sum())
print("Missing angles:", Shots["angle"].isna().sum())

       x     y   distance     angle
0  107.0  53.4  18.669762  0.303090
1   96.5  39.9  23.500213  0.337188
2   88.2  45.5  32.272124  0.243168
3   98.5  49.2  23.385679  0.313313
4  111.2  37.8   9.070832  0.815532
Missing distances: 0
Missing angles: 0


cheking the frames 

In [6]:
Frame = Shots["shot_freeze_frame"].iloc[0]

print("Frame type:", type(Frame))
print("Visible players:", len(Frame))
print("First player:", Frame[0])

Frame type: <class 'numpy.ndarray'>
Visible players: 10
First player: {'location': array([88.8, 54.6]), 'player': {'id': 40724, 'name': 'Florian Wirtz'}, 'position': {'id': 18, 'name': 'Right Attacking Midfield'}, 'teammate': True}


now we chek for visible players 

In [7]:
Shots["visible"] = Shots["shot_freeze_frame"].apply(len)

print(Shots[["visible", "goal"]].head())
print("Minimum visible players:", Shots["visible"].min())
print("Maximum visible players:", Shots["visible"].max())
print("Average visible players:", Shots["visible"].mean())

   visible  goal
0       10     0
1       16     0
2       19     0
3       17     0
4       16     0
Minimum visible players: 1
Maximum visible players: 21
Average visible players: 13.08883564253203


looking for def and GKs

In [8]:
Frame = Shots["shot_freeze_frame"].iloc[0]

for Player in Frame:
    if not Player["teammate"]:
        print(
            "Position:", Player["position"]["name"],
            "| Location:", Player["location"]
        )

Position: Left Center Midfield | Location: [88.9 57.6]
Position: Center Defensive Midfield | Location: [91.1 49.8]
Position: Left Wing Back | Location: [105.   53.9]
Position: Left Center Back | Location: [103.6  51.7]
Position: Center Back | Location: [107.7  47.4]
Position: Right Center Back | Location: [98.8 35.4]
Position: Goalkeeper | Location: [117.6  42.7]


checking for shot on one play first 

In [9]:
ShotX = Shots["x"].iloc[0]
ShotY = Shots["y"].iloc[0]

PlayerX = Frame[0]["location"][0]
PlayerY = Frame[0]["location"][1]

InsideCone = inShotCone(ShotX, ShotY, PlayerX, PlayerY)

print("Shot location:", ShotX, ShotY)
print("Player location:", PlayerX, PlayerY)
print("Inside shot cone:", InsideCone)

Shot location: 107.0 53.4
Player location: 88.8 54.6
Inside shot cone: False


checking for def inside the cone for shot 

In [10]:
DefendersInCone = 0

for Player in Frame:
    if Player["teammate"]:
        continue

    if Player["position"]["name"] == "Goalkeeper":
        continue

    PlayerX = Player["location"][0]
    PlayerY = Player["location"][1]

    if inShotCone(ShotX, ShotY, PlayerX, PlayerY):
        DefendersInCone += 1

print("Defenders inside shot cone:", DefendersInCone)

Defenders inside shot cone: 0


cheking for atck team in the cone 

In [11]:
TeammatesInCone = 0

for Player in Frame:
    if not Player["teammate"]:
        continue

    PlayerX = Player["location"][0]
    PlayerY = Player["location"][1]

    if inShotCone(ShotX, ShotY, PlayerX, PlayerY):
        TeammatesInCone += 1

print("Teammates inside shot cone:", TeammatesInCone)

Teammates inside shot cone: 0


In [12]:
NearestDefender = float("inf")

for Player in Frame:
    if Player["teammate"]:
        continue

    if Player["position"]["name"] == "Goalkeeper":
        continue

    PlayerX = Player["location"][0]
    PlayerY = Player["location"][1]

    Distance = np.hypot(ShotX - PlayerX, ShotY - PlayerY)

    if Distance < NearestDefender:
        NearestDefender = Distance

if NearestDefender == float("inf"):
    NearestDefender = np.nan

print("Nearest defender distance:", NearestDefender)

Nearest defender distance: 2.0615528128088303


looknig for def within 3 yds

In [13]:
DefendersWithin3 = 0

for Player in Frame:
    if Player["teammate"]:
        continue

    if Player["position"]["name"] == "Goalkeeper":
        continue

    PlayerX = Player["location"][0]
    PlayerY = Player["location"][1]

    Distance = np.hypot(ShotX - PlayerX, ShotY - PlayerY)

    if Distance <= 3:
        DefendersWithin3 += 1

print("Defenders within 3 yards:", DefendersWithin3)

Defenders within 3 yards: 1


looking for GK

In [14]:
Keeper = None

for Player in Frame:
    if Player["teammate"]:
        continue

    if Player["position"]["name"] == "Goalkeeper":
        Keeper = Player
        break

if Keeper is None:
    KeeperMissing = 1
else:
    KeeperMissing = 0

print("Goalkeeper missing:", KeeperMissing)

if Keeper is not None:
    print("Goalkeeper location:", Keeper["location"])

Goalkeeper missing: 0
Goalkeeper location: [117.6  42.7]


checking ifg the gk is within the cone 

In [15]:
KeeperInCone = 0

if Keeper is not None:
    KeeperX = Keeper["location"][0]
    KeeperY = Keeper["location"][1]

    if inShotCone(ShotX, ShotY, KeeperX, KeeperY):
        KeeperInCone = 1

print("Goalkeeper inside shot cone:", KeeperInCone)

Goalkeeper inside shot cone: 1


dist to gk

In [16]:
KeeperToGoal = np.nan
KeeperToShooter = np.nan

if Keeper is not None:
    KeeperX = Keeper["location"][0]
    KeeperY = Keeper["location"][1]

    KeeperToGoal = distanceToGoal(KeeperX, KeeperY)
    KeeperToShooter = np.hypot(ShotX - KeeperX, ShotY - KeeperY)

print("Goalkeeper to goal:", KeeperToGoal)
print("Goalkeeper to shooter:", KeeperToShooter)

Goalkeeper to goal: 3.612478373637695
Goalkeeper to shooter: 15.061540425866134


creating frezable frame func

In [17]:
def GetFrameFeatures(Frame, ShotX, ShotY):
    """Calculate player-position features for one shot."""

    Visible = len(Frame)
    DefendersInCone = 0
    TeammatesInCone = 0
    NearestDefender = np.nan
    DefendersWithin3 = 0

    KeeperMissing = 1
    KeeperInCone = 0
    KeeperToGoal = np.nan
    KeeperToShooter = np.nan

    for Player in Frame:
        PlayerX = Player["location"][0]
        PlayerY = Player["location"][1]

        InsideCone = inShotCone(ShotX, ShotY, PlayerX, PlayerY)

        if Player["teammate"]:
            if InsideCone:
                TeammatesInCone += 1
            continue

        if Player["position"]["name"] == "Goalkeeper":
            KeeperMissing = 0

            if InsideCone:
                KeeperInCone = 1

            KeeperToGoal = distanceToGoal(PlayerX, PlayerY)
            KeeperToShooter = np.hypot(ShotX - PlayerX, ShotY - PlayerY)
            continue

        if InsideCone:
            DefendersInCone += 1

        Distance = np.hypot(ShotX - PlayerX, ShotY - PlayerY)

        if np.isnan(NearestDefender) or Distance < NearestDefender:
            NearestDefender = Distance

        if Distance <= 3:
            DefendersWithin3 += 1

    return {
        "visible": Visible,
        "defendersInCone": DefendersInCone,
        "teammatesInCone": TeammatesInCone,
        "nearestDefender": NearestDefender,
        "defendersWithin3": DefendersWithin3,
        "keeperMissing": KeeperMissing,
        "keeperInCone": KeeperInCone,
        "keeperToGoal": KeeperToGoal,
        "keeperToShooter": KeeperToShooter
    }

testing

In [18]:
Features = GetFrameFeatures(Frame, ShotX, ShotY)

for Name, Value in Features.items():
    print(Name, ":", Value)

visible : 10
defendersInCone : 0
teammatesInCone : 0
nearestDefender : 2.0615528128088303
defendersWithin3 : 1
keeperMissing : 0
keeperInCone : 1
keeperToGoal : 3.612478373637695
keeperToShooter : 15.061540425866134


testinbv again for 100 shiot 

In [19]:
for Index in range(100):
    Frame = Shots["shot_freeze_frame"].iloc[Index]
    ShotX = Shots["x"].iloc[Index]
    ShotY = Shots["y"].iloc[Index]

    Features = GetFrameFeatures(Frame, ShotX, ShotY)

    if Index < 5:
        print("Shot:", Index, "| Features:", Features)

print("Shots tested:", 100)

Shot: 0 | Features: {'visible': 10, 'defendersInCone': 0, 'teammatesInCone': 0, 'nearestDefender': np.float64(2.0615528128088303), 'defendersWithin3': 1, 'keeperMissing': 0, 'keeperInCone': 1, 'keeperToGoal': np.float64(3.612478373637695), 'keeperToShooter': np.float64(15.061540425866134)}
Shot: 1 | Features: {'visible': 16, 'defendersInCone': 1, 'teammatesInCone': 0, 'nearestDefender': np.float64(6.319810123729988), 'defendersWithin3': 0, 'keeperMissing': 0, 'keeperInCone': 1, 'keeperToGoal': np.float64(2.2203603311174542), 'keeperToShooter': np.float64(21.300938946440834)}
Shot: 2 | Features: {'visible': 19, 'defendersInCone': 2, 'teammatesInCone': 0, 'nearestDefender': np.float64(5.434151267677408), 'defendersWithin3': 0, 'keeperMissing': 0, 'keeperInCone': 1, 'keeperToGoal': np.float64(2.4331050121192934), 'keeperToShooter': np.float64(29.98616347584331)}
Shot: 3 | Features: {'visible': 17, 'defendersInCone': 4, 'teammatesInCone': 1, 'nearestDefender': np.float64(9.631718434422806)

testing for all shots now 

In [20]:
FrameFeatures = []

for Index, Shot in Shots.iterrows():
    Features = GetFrameFeatures(
        Shot["shot_freeze_frame"],
        Shot["x"],
        Shot["y"]
    )

    FrameFeatures.append(Features)

    if (Index + 1) % 10000 == 0:
        print("Processed:", Index + 1, "of", len(Shots))

FrameFeatures = pd.DataFrame(FrameFeatures)

print("Total feature rows:", len(FrameFeatures))
print("Feature columns:", len(FrameFeatures.columns))

Processed: 10000 of 99746
Processed: 20000 of 99746
Processed: 30000 of 99746
Processed: 40000 of 99746
Processed: 50000 of 99746
Processed: 60000 of 99746
Processed: 70000 of 99746
Processed: 80000 of 99746
Processed: 90000 of 99746
Total feature rows: 99746
Feature columns: 9


In [21]:
Shots = pd.concat(
    [
        Shots.reset_index(drop=True),
        FrameFeatures.drop(columns=["visible"]).reset_index(drop=True)
    ],
    axis=1
)

print("Total shots:", len(Shots))
print("Total columns:", len(Shots.columns))

Total shots: 99746
Total columns: 61


In [22]:
Shots = Shots.loc[:, ~Shots.columns.duplicated()].copy()

print("Total shots:", len(Shots))
print("Total columns:", len(Shots.columns))
print("Duplicate columns:", Shots.columns[Shots.columns.duplicated()].tolist())

Total shots: 99746
Total columns: 61
Duplicate columns: []


In [23]:
Shots["ConeGroup"] = Shots["defendersInCone"].clip(upper=3)

ConeCheck = Shots.groupby("ConeGroup")["goal"].agg(
    ["count", "mean"]
)

ConeCheck["mean"] = ConeCheck["mean"] * 100

print(ConeCheck)

           count       mean
ConeGroup                  
0          47304  14.671064
1          33886   6.698932
2          11506   5.431949
3           7050   5.645390


cehcking for missing value in our feature

In [24]:
FeatureNames = [
    "visible",
    "defendersInCone",
    "teammatesInCone",
    "nearestDefender",
    "defendersWithin3",
    "keeperMissing",
    "keeperInCone",
    "keeperToGoal",
    "keeperToShooter",
    "distance",
    "angle"
]

print(Shots[FeatureNames].isna().sum())

visible               0
defendersInCone       0
teammatesInCone       0
nearestDefender       8
defendersWithin3      0
keeperMissing         0
keeperInCone          0
keeperToGoal        100
keeperToShooter     100
distance              0
angle                 0
dtype: int64


In [25]:
print("Total columns:", len(Shots.columns))

print("Duplicate columns:")
print(Shots.columns[Shots.columns.duplicated()].tolist())

print("keeperMissing count:", (Shots.columns == "keeperMissing").sum())
print("keeperToGoal count:", (Shots.columns == "keeperToGoal").sum())

Total columns: 62
Duplicate columns:
[]
keeperMissing count: 1
keeperToGoal count: 1


cheking gk missing dist matches keepermissing

In [26]:
print("Goalkeepers missing:", Shots["keeperMissing"].sum())

print(
    "Missing keeper-to-goal distances:",
    Shots["keeperToGoal"].isna().sum()
)

print(
    "Missing keeper-to-shooter distances:",
    Shots["keeperToShooter"].isna().sum()
)

print(
    "Missing goalkeeper consistency:",
    (
        (Shots["keeperMissing"] == 1)
        == Shots["keeperToGoal"].isna()
    ).all()
)

Goalkeepers missing: 100
Missing keeper-to-goal distances: 100
Missing keeper-to-shooter distances: 100
Missing goalkeeper consistency: True


In [27]:
FlagColumns = [
    "under_pressure",
    "shot_first_time",
    "shot_one_on_one",
    "shot_open_goal",
    "shot_aerial_won",
    "shot_follows_dribble",
    "shot_redirect"
]

for Column in FlagColumns:
    print(Column, ":", Column in Shots.columns)

under_pressure : True
shot_first_time : True
shot_one_on_one : True
shot_open_goal : True
shot_aerial_won : True
shot_follows_dribble : True
shot_redirect : True


In [28]:
BaseColumns = [
    "id", "match_id", "test", "fold", "goal", "x", "y"
]

GeometryColumns = [
    "distance", "angle"
]

FrameColumns = [
    "visible",
    "defendersInCone",
    "teammatesInCone",
    "nearestDefender",
    "defendersWithin3",
    "keeperMissing",
    "keeperInCone",
    "keeperToGoal",
    "keeperToShooter"
]

DummyColumns = Encoded.columns.tolist()

FinalColumns = BaseColumns + GeometryColumns + FrameColumns + DummyColumns + FlagColumns

FinalShots = Shots[FinalColumns].copy()

print("Total shots:", len(FinalShots))
print("Total columns:", len(FinalShots.columns))
print("Duplicate columns:", FinalShots.columns.duplicated().sum())
print("Missing required columns:", set(FinalColumns) - set(Shots.columns))

Total shots: 99746
Total columns: 49
Duplicate columns: 0
Missing required columns: set()


saving shots feat in data

In [29]:
from pathlib import Path

OutputPath = Path("../data/features/shots_features.parquet")

OutputPath.parent.mkdir(parents=True, exist_ok=True)

FinalShots.to_parquet(OutputPath, index=False)

SavedShots = pd.read_parquet(OutputPath)

print("Saved rows:", len(SavedShots))
print("Saved columns:", len(SavedShots.columns))
print("Matches original:", SavedShots.equals(FinalShots))
print("Missing values in flags:", SavedShots[FlagColumns].isna().sum().sum())

Saved rows: 99746
Saved columns: 49
Matches original: True
Missing values in flags: 0


verifying the saved data now

In [30]:
SavedShots = pd.read_parquet(OutputPath)

print("Rows:", len(SavedShots))
print("Columns:", len(SavedShots.columns))
print("Matches original:", SavedShots.equals(FinalShots))
print("Duplicate columns:", SavedShots.columns.duplicated().sum())

Rows: 99746
Columns: 49
Matches original: True
Duplicate columns: 0


In [31]:
for Column in FlagColumns:
    print(Column)
    print(Shots[Column].value_counts(dropna=False))
    print()

under_pressure
under_pressure
0    74777
1    24969
Name: count, dtype: int64

shot_first_time
shot_first_time
0    68502
1    31244
Name: count, dtype: int64

shot_one_on_one
shot_one_on_one
0    94815
1     4931
Name: count, dtype: int64

shot_open_goal
shot_open_goal
0    98694
1     1052
Name: count, dtype: int64

shot_aerial_won
shot_aerial_won
0    91247
1     8499
Name: count, dtype: int64

shot_follows_dribble
shot_follows_dribble
0    99682
1       64
Name: count, dtype: int64

shot_redirect
shot_redirect
0    99548
1      198
Name: count, dtype: int64



In [32]:
SavedShots = pd.read_parquet("../data/features/shots_features.parquet")

print("Rows:", len(SavedShots))
print("Columns:", len(SavedShots.columns))
print("Missing flags:", [
    Column for Column in FlagColumns
    if Column not in SavedShots.columns
])
print("Duplicate columns:", SavedShots.columns.duplicated().sum())

Rows: 99746
Columns: 49
Missing flags: []
Duplicate columns: 0
